# UCI Benchmark: FCM, PCM, PFCM, DPC-FCM, eFCM, and DA-FCM

This notebook evaluates all benchmark methods on the UCI benchmark set: Iris, Wine, Breast Cancer, Seeds, Glass, Ecoli, Yeast, and Image Segmentation.

The executable implementations live in `uci_benchmark_algorithms.py` to keep this notebook readable. The module is self-contained and uses only NumPy and pandas; it does not require scikit-learn.

Generated files are written under `results/`:
- `all_algorithms_uci_benchmark_summary.csv`: best run per dataset-method pair
- `all_algorithms_uci_all_runs.csv`: every initialization


## Methods Included

- `FCM`: standard Fuzzy C-Means with global `m = 2.0`.
- `PCM`: Possibilistic C-Means using FCM-derived scale values.
- `PFCM`: Possibilistic Fuzzy C-Means with `m = nu = 2.0`, `a = b = 1.0`.
- `DPC-FCM`: density-peak initialized FCM using the same Gaussian density idea as DA-FCM only for initialization.
- `eFCM`: entropy-regularized FCM using softmax membership updates with automatic lambda.
- `DA-FCM`: density-adaptive FCM with `m_low = 1.2`, `m_high = 4.0`, `alpha = 5.0`.

Metrics reported: PC, PE, Silhouette, Purity, NMI, ARI, iterations, runtime, and method-specific diagnostic fields where available.

In [ ]:
from pathlib import Path

import pandas as pd

from uci_benchmark_algorithms import (
    DEFAULT_MAX_ITER,
    DEFAULT_RANDOM_STATE,
    DEFAULT_TOL,
    benchmark_all_methods,
    load_uci_datasets,
)

ROOT = Path.cwd()
DATA_DIR = ROOT / "uci_data"
RESULT_DIR = ROOT / "results"
RESULT_DIR.mkdir(exist_ok=True)

METHODS = ["FCM", "PCM", "PFCM", "DPC-FCM", "eFCM", "DA-FCM"]
N_INIT = 5
RANDOM_STATE = DEFAULT_RANDOM_STATE
MAX_ITER = DEFAULT_MAX_ITER
TOL = DEFAULT_TOL

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)


In [ ]:
datasets = load_uci_datasets(DATA_DIR)

dataset_table = pd.DataFrame([
    {
        "Dataset": ds["name"],
        "Samples": ds["samples"],
        "Features": ds["features"],
        "Classes/K": ds["classes"],
        "Characteristics": ds["characteristics"],
    }
    for ds in datasets
])
dataset_table

In [ ]:
summary_df, all_runs_df = benchmark_all_methods(
    datasets,
    methods=METHODS,
    n_init=N_INIT,
    random_state=RANDOM_STATE,
    max_iter=MAX_ITER,
    tol=TOL,
)

summary_path = RESULT_DIR / "all_algorithms_uci_benchmark_summary.csv"
all_runs_path = RESULT_DIR / "all_algorithms_uci_all_runs.csv"
summary_df.to_csv(summary_path, index=False)
all_runs_df.to_csv(all_runs_path, index=False)

print(f"Saved summary: {summary_path}")
print(f"Saved all runs: {all_runs_path}")

In [ ]:
summary_columns = [
    "dataset", "method", "samples", "features", "classes", "n_init", "best_seed",
    "converged", "iterations", "total_time_s", "pc", "pe", "sil", "pur", "nmi", "ari", "objective",
]
summary_df[summary_columns].sort_values(["dataset", "method"])

In [ ]:
# PC and PE are the main fuzzy-membership quality metrics used in the paper.
pc_table = summary_df.pivot(index="dataset", columns="method", values="pc").reindex(columns=METHODS)
pe_table = summary_df.pivot(index="dataset", columns="method", values="pe").reindex(columns=METHODS)

display(pc_table.round(4).style.highlight_max(axis=1))
display(pe_table.round(4).style.highlight_min(axis=1))

In [ ]:
# Accuracy-style metrics against the available class labels.
for metric in ["sil", "pur", "nmi", "ari"]:
    table = summary_df.pivot(index="dataset", columns="method", values=metric).reindex(columns=METHODS)
    print(metric.upper())
    display(table.round(4).style.highlight_max(axis=1))

In [ ]:
# Runtime and convergence overview.
runtime_table = (summary_df.assign(time_ms=summary_df["total_time_s"] * 1000)
                 .pivot(index="dataset", columns="method", values="time_ms")
                 .reindex(columns=METHODS))
iteration_table = summary_df.pivot(index="dataset", columns="method", values="iterations").reindex(columns=METHODS)

display(iteration_table.astype(int).style.highlight_min(axis=1))
display(runtime_table.round(2).style.highlight_min(axis=1))

In [ ]:
# Optional: inspect every initialization for stability/debugging.
all_runs_df[[
    "dataset", "method", "init_index", "seed", "converged", "iterations", "total_time_s", "objective", "pc", "pe", "sil", "pur", "nmi", "ari"
]].sort_values(["dataset", "method", "init_index"])